# Impact of solver discrepancy on the classification

In [1]:
from typing import Callable
from types import SimpleNamespace
from pipe import select

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import linalg

import plotly.express as px
import plotly.graph_objects as go

In [2]:
SEED = 123
np.random.seed(SEED)

We will take stiff dynamical system to illustrate why it is important to account for the solver error in the trajectory classification. The physical anology for the problem may be a ball moving on the saddle near equilibrium.

In [3]:
# system params
systems = [
    {
        "lambda1": -35.,
        "lambda2": 0.5,
        "e1": np.array([1., 0]),
        "e2": np.array([0., 1.]),
    },
    {
        "lambda1": -10.,
        "lambda2": 0.6,
        "e1": np.array([1., 0]),
        "e2": np.array([0., 1.]),
    }
]
systems = list(systems | select(lambda d: SimpleNamespace(**d)))

In [4]:
def transition_matrix(sys_c: SimpleNamespace):
    e1, e2 = list(
        [sys_c.e1, sys_c.e2] |
        select(lambda e: e / linalg.norm(e)) |
        select(lambda e: e[:, None])
    )
    U = np.hstack([e1, e2])
    return U

# building matrix A in our ODE x' = A @ x
def system_matrix(
    sys_c: SimpleNamespace
):
    L = np.diag([sys_c.lambda1, sys_c.lambda2])
    U = transition_matrix(sys_c)
    A = (U @ L @ U.T)
    return A

In [5]:
def compute_systems_matrices():
    sys_matrices = list(systems | select(lambda d: system_matrix(d)))
    print(*sys_matrices, sep='\n')
compute_systems_matrices()

[[-35.    0. ]
 [  0.    0.5]]
[[-10.    0. ]
 [  0.    0.6]]


In [6]:
# exact solution
def exact_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    if t.size == 1:
        return x0[None, :]

    U = transition_matrix(sys_c)
    x0_trans = U.T @ x0
    l = np.repeat(np.array([sys_c.lambda1, sys_c.lambda2])[:, None], t.shape[0], axis=1)
    sol = x0_trans * np.exp(l * t).T
    sol = U @ sol.T
    return sol.T

In [7]:
# numerical eurler solution
def euler_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    if t.size == 1:
        return x0[None, :]

    # TODO: use functional programming
    A = system_matrix(sys_c)
    traj = np.zeros([t.shape[0], x0.shape[0]])
    traj[0] = x0

    for i in range(1, t.shape[0]):
        traj[i] = traj[i-1] + (t[i] - t[i-1]) * (A @ traj[i-1])

    return traj

In [8]:
# numerical solution
def RK_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    if t.size == 1:
        return x0[None, :]

    A = system_matrix(sys_c)
    return solve_ivp(
        lambda t, x: A.dot(x),
        [t[0], t[-1]],
        x0,
        method="RK23",
        t_eval=t
    ).y.T

In [9]:
x0 = np.array([1e3, 1.])

In [10]:
T = 0.5
delta_t = 0.5e-1
t = np.linspace(0., T, int(T / delta_t))

In [11]:
def vizualize_solutions():
    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    # solve sytems with euler
    euler_sol = list(
        systems |
        select(lambda s: euler_solver(t, s, x0))
    )
    # solve sytems with RK23
    RK23_sol = list(
        systems |
        select(lambda s: RK_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
        "euler": euler_sol,
        "RK23": RK23_sol
    }

    fig = go.Figure()
    for sol_name, sol in solutions.items():
        for i, sys_sol in enumerate(sol):
            line = dict(dash="dash") if sol_name.count("true") == 0 else None
            fig.add_trace(
                go.Scatter(
                    x=sys_sol[:, 0],
                    y=sys_sol[:, 1],
                    mode='lines',
                    name=f"{sol_name}_{i}",
                    line=line
                )
            )
    fig.add_trace(
        go.Scatter(
            x=[x0[0]],
            y=[x0[1]],
            name="start"
        )
    )
    fig.show()

vizualize_solutions()

## Classification

Let's classify noisy trajectory from system 0 and measure accuracy

In [12]:
# we choose the model with the least mse
def cls_loss(sample_traj: np.ndarray, mean_traj: np.ndarray):
    return np.linalg.norm(sample_traj - mean_traj, ord=2, axis=1).mean()

In [13]:
NUM_SAMPLES = 1000
sigma = 0.05

In [14]:
def simple_classification():
    np.random.seed(SEED)
    accuracy_df = pd.DataFrame(
        {
            "accuracy": np.zeros([3]),
            "method": ["true", "euler", "RK23"],
        }
    )
    traj_0 = exact_solver(t, systems[0], x0)
    num_traj_points = traj_0.shape[0]

    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda s: exact_solver(t, s, x0))
    )
    # solve sytems with euler
    euler_sol = list(
        systems |
        select(lambda s: euler_solver(t, s, x0))
    )
    # solve sytems with RK23
    RK23_sol = list(
        systems |
        select(lambda s: RK_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
        "euler": euler_sol,
        "RK23": RK23_sol
    }
    
    for i in range(NUM_SAMPLES):
        sample_traj = traj_0 + sigma * np.random.randn(num_traj_points, 2)

        if i == 0:
            fig = go.Figure()
            for sol_name, sol in solutions.items():
                for i, sys_sol in enumerate(sol):
                    line = dict(dash="dash") if sol_name.count("true") == 0 else None
                    fig.add_trace(
                        go.Scatter(
                            x=sys_sol[:, 0],
                            y=sys_sol[:, 1],
                            mode='lines',
                            name=f"{sol_name}_{i}",
                            line=line
                        )
                    )
            fig.add_trace(
                go.Scatter(
                    x=sample_traj[:, 0],
                    y=sample_traj[:, 1],
                    name="samples",
                    mode='lines'
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[x0[0]],
                    y=[x0[1]],
                    name="start"
                )
            )
            fig.show()

        for sol_name, sol in solutions.items():
            sol_losses = list(
                range(2) |
                select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
            )
            if sol_losses[0] < sol_losses[1]:
                accuracy_df.loc[
                    accuracy_df["method"] == sol_name, 
                    "accuracy"
                ] += 1.
    accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES

    return accuracy_df

accuracy_df = simple_classification()
accuracy_df

,accuracy,method
0,1.0,true
1,0.0,euler
2,1.0,RK23


## Segmented classification

In [15]:
accuracy_df["segment_size"] = t.size

In [16]:
def segmented_solution(
    t: np.ndarray, sys_c: SimpleNamespace, traj: np.ndarray, 
    segment_size: int, solver_f: Callable
):
    """The trajectory is split into segments of length segment_size.
        Each segment is integrated separetely. The overall loss is a sum of
        the losses over the segments. The initial point of each segment is taken
        as it is, we do not account for small noise here.
    """
    sol = []
    cur_start = 0
    while cur_start < t.size:
        cur_x0 = traj[cur_start]
        sol.append(
            solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                cur_x0
            )
        )
        cur_start = cur_start + segment_size

    return np.concat(sol)

In [ ]:
segment_sizes = np.array([1, 3, 5])

In [18]:
def segmented_classification():
    accuracy_df = []
    for seg_size in segment_sizes:
        np.random.seed(SEED)
        cur_accuracy_df = pd.DataFrame(
            {
                "accuracy": np.zeros([3]),
                "method": ["true", "euler", "RK23"],
                "segment_size": np.full([3], seg_size)
            }
        )
        traj_0 = exact_solver(t, systems[0], x0)
        num_traj_points = traj_0.shape[0]
        
        for i in range(NUM_SAMPLES):
            sample_traj = traj_0 + sigma * np.random.randn(num_traj_points, 2)

            true_seg_solutions = list(
                systems |
                select(lambda s: segmented_solution(t, s, sample_traj, seg_size, exact_solver))
            )
            rk_seg_solutions = list(
                systems |
                select(lambda s: segmented_solution(t, s, sample_traj, seg_size, RK_solver))
            )
            euler_seg_solutions = list(
                systems |
                select(lambda s: segmented_solution(t, s, sample_traj, seg_size, euler_solver))
            )
            solutions = {
                "true": true_seg_solutions,
                "euler": euler_seg_solutions,
                "RK23": rk_seg_solutions
            }

            if i == 0:
                print("Segment size =", seg_size)
                fig = go.Figure()
                fig.add_trace(
                    go.Scatter(
                        x=traj_0[:, 0],
                        y=traj_0[:, 1],
                        mode='lines',
                        name="true_0"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=sample_traj[:, 0],
                        y=sample_traj[:, 1],
                        mode='lines',
                        name="samples_true"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=true_seg_solutions[0][:, 0],
                        y=true_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_true"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=rk_seg_solutions[0][:, 0],
                        y=rk_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_rk",
                        line=dict(dash="dash")
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=euler_seg_solutions[0][:, 0],
                        y=euler_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_euler",
                        line=dict(dash="dash")
                    )
                )
                fig.show()

            for sol_name, sol in solutions.items():
                sol_losses = list(
                    range(2) |
                    select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
                )
                if sol_losses[0] < sol_losses[1]:
                    cur_accuracy_df.loc[
                        cur_accuracy_df["method"] == sol_name, 
                        "accuracy"
                    ] += 1.
        cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
        accuracy_df.append(cur_accuracy_df)

    accuracy_df = pd.concat(accuracy_df)
    return accuracy_df

accuracy_df = pd.concat([accuracy_df, segmented_classification()])
accuracy_df

Segment size = 3


Segment size = 5


,accuracy,method,segment_size
0,1.0,true,10
1,0.0,euler,10
2,1.0,RK23,10
0,1.0,true,3
1,0.0,euler,3
2,1.0,RK23,3
0,1.0,true,5
1,0.0,euler,5
2,1.0,RK23,5


## Vizualize the results

In [21]:
px.bar(
    accuracy_df,
    x="segment_size",
    y="accuracy",
    color="method",
    barmode='group'
)